In [1]:
import pandas as pd
import numpy as np
import time
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

df = pd.read_csv('../data/processed/iris_cleaned.csv')
X = df[['SepalLengthCm', 'SepalWidthCm', 'PetalLengthCm', 'PetalWidthCm']]
y = df['Species']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

models = {
    'KNN':           KNeighborsClassifier(n_neighbors=5),
    'Decision Tree': DecisionTreeClassifier(random_state=42),
    'SVM':           SVC(kernel='rbf', random_state=42),
}

print("=" * 60)
print(f"{'Model':<15} {'Train Time':>12} {'Predict Time':>14} {'Accuracy':>10}")
print("-" * 60)

timing_results = {}
for name, model in models.items():
    start = time.perf_counter()
    model.fit(X_train, y_train)
    train_time = time.perf_counter() - start
    start = time.perf_counter()
    y_pred = model.predict(X_test)
    pred_time = time.perf_counter() - start
    acc = accuracy_score(y_test, y_pred)
    timing_results[name] = {'train': train_time, 'predict': pred_time, 'acc': acc}
    print(f"{name:<15} {train_time*1000:>10.2f}ms {pred_time*1000:>12.2f}ms {acc:>10.2%}")
print("=" * 60)

Model             Train Time   Predict Time   Accuracy
------------------------------------------------------------
KNN                  10.96ms        11.68ms    100.00%
Decision Tree        11.34ms         4.11ms    100.00%
SVM                  12.00ms         6.53ms    100.00%


In [2]:
df_large = pd.concat([df] * 100, ignore_index=True)
print(f"Large dataset size: {len(df_large)} rows")
X_large = df_large[['SepalLengthCm', 'SepalWidthCm', 'PetalLengthCm', 'PetalWidthCm']]
y_large = df_large['Species']

print("\nSTRESS TEST — Large Dataset")
print("=" * 40)
for name, model in models.items():
    start = time.perf_counter()
    model.fit(X_large, y_large)
    duration = time.perf_counter() - start
    print(f"{name:<15} trained on 15,000 rows in {duration*1000:.2f}ms ✅")
print("\n✅ No crashes!")

Large dataset size: 15000 rows

STRESS TEST — Large Dataset
KNN             trained on 15,000 rows in 100.39ms ✅
Decision Tree   trained on 15,000 rows in 68.83ms ✅
SVM             trained on 15,000 rows in 1358.82ms ✅

✅ No crashes!


In [3]:
import joblib

cache_path = '../data/processed/svm_model_cache.pkl'
joblib.dump(models['SVM'], cache_path)
print(f"✅ Model cached to {cache_path}")

start = time.perf_counter()
cached_model = joblib.load(cache_path)
load_time = time.perf_counter() - start
print(f"✅ Loaded from cache in {load_time*1000:.2f}ms")

test_input = pd.DataFrame([[5.1, 3.5, 1.4, 0.2]],
             columns=['SepalLengthCm','SepalWidthCm','PetalLengthCm','PetalWidthCm'])
print(f"✅ Cached model prediction: {cached_model.predict(test_input)[0]}")

✅ Model cached to ../data/processed/svm_model_cache.pkl
✅ Loaded from cache in 7.67ms
✅ Cached model prediction: Iris-setosa


In [4]:
print("\nMEMORY USAGE")
print("=" * 40)
print(f"Original df memory:   {df.memory_usage(deep=True).sum() / 1024:.2f} KB")

df_opt = df.copy()
for col in ['SepalLengthCm','SepalWidthCm','PetalLengthCm','PetalWidthCm']:
    df_opt[col] = df_opt[col].astype('float32')
df_opt['Species'] = df_opt['Species'].astype('category')

print(f"Optimized df memory:  {df_opt.memory_usage(deep=True).sum() / 1024:.2f} KB")
print(f"✅ Memory reduced!")


MEMORY USAGE
Original df memory:   15.12 KB
Optimized df memory:  3.97 KB
✅ Memory reduced!
